# Midterm 1: run the data generator in Colab

Use this notebook if you cannot run Python or git on your own machine. It only **generates and shows your data**. You write your answers in `ANSWERS.md` on **your fork on github.com** (open the file, click the pencil icon, edit, *Commit changes*), then submit the commit URL in Canvas. See the README for the rules.

1. Fill in `GITHUB_USERNAME` (the account that owns your fork) and `CODE` (`PRACTICE` to rehearse; the real code is on the board during the exam).
2. Run the three code cells in order.

In [1]:
GITHUB_USERNAME = "s23069417"   # e.g. "aigerim-k"
CODE = "MAPLE-66"      # replace with the session code during the exam
assert GITHUB_USERNAME, "fill in GITHUB_USERNAME first"

In [6]:
import os

if not os.path.exists("midterm-1-AI-course"):
    !git clone --quiet https://github.com/unreal-kz/midterm-1-AI-course.git
%cd midterm-1-AI-course
!python make_my_data.py --github s23069417 --code MAPLE-66

/content/midterm-1-AI-course/midterm-1-AI-course/midterm-1-AI-course
Wrote 3 files to my_data/ · fingerprint 1cdb3e29
Next: open my_data/DATA.md


In [7]:
import pandas as pd
from IPython.display import Markdown, display

display(Markdown(open("my_data/DATA.md", encoding="utf-8").read()))
ranks = pd.read_csv("my_data/retrieval_ranks.csv")   # Part 1
docs = pd.read_csv("my_data/doc_results.csv")        # Part 2
display(ranks.head(), docs)

# Midterm 1: your data

GitHub username: `s23069417` · session code: `maple-66` · generator v1

**Fingerprint: `1cdb3e29`**: copy it into the first lines of `ANSWERS.md`. All numbers below are synthetic: this is an exam, not a measurement.

## Part 1: inputs

`retrieval_ranks.csv`: 60 test questions (20 EN, 20 RU, 20 KK). `rank_A` and `rank_B` give the position of the correct chunk in the top 10 retrieved under config A and config B; **11 means the correct chunk was not in the top 10.**

Production traffic and Model M token counts (tokens per item, as counted by Model M's tokenizer):

| language | share of traffic | question tokens | answer tokens | chunk A tokens | chunk B tokens |
|---|---|---|---|---|---|
| EN | 45 % | 25 | 102 | 243 | 63 |
| RU | 28 % | 41 | 168 | 401 | 104 |
| KK | 27 % | 71 | 291 | 693 | 180 |

- Requests per day: **11,500**, in the traffic shares above.
- System prompt: **294** tokens in every request.
- Price: **$1** per million input tokens, **$6** per million output tokens.
- Annual budget for this service (model calls only): **$12,600**.

## Part 2: inputs

`doc_results.csv`: for each pipeline, language and scan condition, `n_docs` test invoices and `n_correct` = invoices whose number, date, seller ID and total were all extracted exactly (strict scorer, as in Lab 04).

- **Pipeline H**: hosted vision-language API in a data centre abroad, pay per token; each page is sent as an image.
- **Pipeline L**: open vision-language model on your own server in Kazakhstan, fixed monthly cost.

- Volume: **5,500** scanned pages per day. In **13 %** of them an individual's name and ИИН are printed (sole proprietors, receipts).
- Page image: **1654 × 2339** pixels. Pipeline H counts an image as ⌈width ÷ 28⌉ × ⌈height ÷ 28⌉ input tokens and returns about **291** output tokens per page (the JSON).
- Pipeline H price: **$2** per million input tokens, **$10** per million output tokens.
- Pipeline L: one server costs **$450** per month and handles up to **4,000** pages per day; add servers as needed.

## Part 3: your five questions

Answer each in `ANSWERS.md`: one **Number** line and one **Why** line of at most 40 words.

### Q1

One complaint sentence, measured three ways (synthetic numbers):

| | characters | UTF-8 bytes | tokens (Model M) |
|---|---|---|---|
| EN | 292 | 292 | 56 |
| RU | 304 | 570 | 92 |
| KK | 333 | 629 | 160 |

Compute the KK÷EN ratio for characters, for bytes and for tokens. **Number:** the token ratio divided by the byte ratio (2 decimals). **Why** (at most 40 words): what does this number say about where the Kazakh premium comes from, the encoding or the tokenizer?

### Q2

A teammate sets the chunk size to **512 tokens, counted with cl100k**. The retrieval model (e5-small) reads at most 512 of *its own* tokens. Assume e5 tokens grow in proportion to cl100k tokens at the rate measured in Lecture 4.1 (a chunker told 256 cl100k tokens produced chunks of 263 / 128 / 76 e5 tokens for EN / RU / KK), and that every chunk is exactly the full size.

**Number:** the share of an English chunk, in percent (1 decimal), that the embedding never sees. **Why** (at most 40 words): which of the three languages are cut, and what happens to the cut part?

### Q3

A language model gives these next-token scores for the prompt "The branch is closed on":

| token | score |
|---|---|
| Monday | 4.9 |
| Friday | 3.4 |
| Sunday | 2.8 |
| holiday | 2.4 |
| Easter | 1.0 |

Apply softmax at temperature T = 2. **Number:** the probability of the top token, in percent (1 decimal). **Why** (at most 40 words): what did changing T do to the ranking of the tokens, and to the set of tokens that can be sampled?

### Q4

Two models answer the same 100 factual questions (1 point per correct answer on the public leaderboard):

| | correct | wrong | "I don't know" |
|---|---|---|---|
| Model X | 66 | 34 | 0 |
| Model Y | 62 | 17 | 21 |

Now score them again with a penalty: +1 per correct answer, −1 per wrong answer, 0 for "I don't know". **Number:** Y's penalised score minus X's (negative if X is ahead). **Why** (at most 40 words): which model does the public leaderboard prefer, and what does that teach a model to do?

### Q5

A support chat adds **101 new tokens per turn** (customer message plus assistant reply) and has **19 turns**. The model keeps no state, so the full history is sent again on every turn (Lecture 1.2). As in the Lecture 1.2 table (100, 200, 300, ... tokens billed on turns 1, 2, 3, ...), turn *i* is billed for the tokens of all *i* turns so far, including its own. **Number:** the total input tokens billed over the whole conversation. **Why** (at most 40 words): how many times has the first message been paid for by the last turn, and what does prompt caching change?


,qid,lang,rank_A,rank_B
0,q01,EN,1,3
1,q02,KK,1,1
2,q03,RU,1,1
3,q04,KK,1,3
4,q05,KK,11,7


,pipeline,lang,condition,n_docs,n_correct
0,H,EN,clean,30,28
1,H,EN,degraded,30,27
2,H,RU,clean,30,29
3,H,RU,degraded,30,28
4,H,KK,clean,30,27
5,H,KK,degraded,30,26
6,L,EN,clean,30,28
7,L,EN,degraded,30,25
8,L,RU,clean,30,27
9,L,RU,degraded,30,23


In [8]:
import math
Z = 1.96
def wilson(h, n):
    p = h / n; d = 1 + Z*Z/n
    c = (p + Z*Z/(2*n)) / d
    w = Z * math.sqrt(p*(1-p)/n + Z*Z/(4*n*n)) / d
    return round(c - w, 3), round(c + w, 3)

# ---- 1a ----
n = len(ranks)
for cfg in "AB":
    r = ranks["rank_" + cfg]
    rec = [round((r <= k).sum() / n, 3) for k in (1, 3, 5)]
    mrr = round((1 / r[r <= 10]).sum() / n, 3)
    print("1a", cfg, "recall@1,3,5 =", rec, "MRR@10 =", mrr)

# ---- 1b ----
for cfg, k in [("A", 3), ("A", 5), ("B", 3), ("B", 5)]:
    h = int((ranks["rank_" + cfg] <= k).sum())
    print("1b", cfg, k, f"{h}/{n}", round(h / n, 3), wilson(h, n))

# ---- 1c ----
share = {"EN": .45, "RU": .28, "KK": .27}
q = {"EN": 25, "RU": 41, "KK": 71}
a = {"EN": 102, "RU": 168, "KK": 291}
ch = {"A": {"EN": 243, "RU": 401, "KK": 693}, "B": {"EN": 63, "RU": 104, "KK": 180}}
def cost(cfg, k):
    s = 0
    for l in share:
        tin = 294 + q[l] + k * ch[cfg][l]
        s += share[l] * (tin * 1 + a[l] * 6) / 1e6
    return s * 11500 * 365
for cfg, k in [("A", 3), ("A", 5), ("B", 3), ("B", 5)]:
    print("1c", cfg, k, round(cost(cfg, k)))

# ---- 2a ----
for p in "HL":
    d = docs[docs.pipeline == p].copy()
    N, C = int(d.n_docs.sum()), int(d.n_correct.sum())
    d["acc"] = d.n_correct / d.n_docs
    w = d.sort_values("acc").iloc[0]
    print("2a", p, "overall", C, "/", N, round(C / N, 3),
          "| weakest:", w.lang, w.condition, round(w.acc, 3),
          wilson(int(w.n_correct), int(w.n_docs)))
img = math.ceil(1654/28) * math.ceil(2339/28)
print("2a cost H", round(5500*365*(img*2 + 291*10)/1e6),
      "| cost L", math.ceil(5500/4000)*450*12,
      "| PII pages/year", round(5500*365*0.13))

1a A recall@1,3,5 = [np.float64(0.483), np.float64(0.583), np.float64(0.667)] MRR@10 = 0.542
1a B recall@1,3,5 = [np.float64(0.45), np.float64(0.567), np.float64(0.817)] MRR@10 = 0.556
1b A 3 35/60 0.583 (0.457, 0.699)
1b A 5 40/60 0.667 (0.541, 0.773)
1b B 3 34/60 0.567 (0.441, 0.684)
1b B 5 49/60 0.817 (0.701, 0.894)
1c A 3 10876
1c A 5 14308
1c B 3 7065
1c B 5 7956
2a H overall 165 / 180 0.917 | weakest: KK degraded 0.867 (0.703, 0.947)
2a L overall 147 / 180 0.817 | weakest: KK degraded 0.733 (0.556, 0.858)
2a cost H 26077 | cost L 10800 | PII pages/year 260975


In [9]:
import math

# Q1
tok = 160/56; byt = 629/292
print("Q1", round(tok/byt, 2))

# Q2
e5 = 512 * 263/256
print("Q2", round((e5-512)/e5*100, 1), "%")

# Q3
s = [4.9, 3.4, 2.8, 2.4, 1.0]
e = [math.exp(x/2) for x in s]
print("Q3", round(e[0]/sum(e)*100, 1), "%")

# Q4
print("Q4", (62-17) - (66-34))

# Q5
print("Q5", 101 * sum(range(1, 20)))

Q1 1.33
Q2 2.7 %
Q3 44.4 %
Q4 13
Q5 19190


`ranks` and `docs` are ready for you to compute with. Remember the **fingerprint** at the top of the data sheet: it goes into the first lines of `ANSWERS.md`.